# Module 4b: Spark vs Ray Comparison

**DSC 232R - Big Data Analysis Using Spark**

This notebook compares Spark and Ray:
1. API and programming model differences
2. Performance characteristics
3. Use case suitability
4. When to use each framework

## Key Takeaways

- **Spark** excels at ETL, SQL queries, and batch processing
- **Ray** excels at ML training, custom tasks, and stateful computation
- Choose based on workload: not "Spark OR Ray" but "Spark AND Ray"

In [ ]:
import ray
import numpy as np
import pandas as pd
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize both frameworks
if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, logging_level="WARNING")

spark = SparkSession.builder \
    .appName("SparkRayComparison") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Ray version: {ray.__version__}")
print(f"Spark version: {spark.version}")

---

## 1. API Comparison

### Creating Distributed Data

In [ ]:
# Generate sample data
np.random.seed(42)
n_rows = 100000

data = pd.DataFrame({
    "id": range(n_rows),
    "value": np.random.randn(n_rows),
    "category": np.random.choice(["A", "B", "C"], n_rows),
    "timestamp": pd.date_range("2023-01-01", periods=n_rows, freq="s")
})

print(f"Sample data: {len(data):,} rows")

In [ ]:
# SPARK: Create DataFrame
spark_df = spark.createDataFrame(data)
print("Spark DataFrame:")
spark_df.show(3)
print(f"Partitions: {spark_df.rdd.getNumPartitions()}")

In [ ]:
# RAY: Create Dataset
ray_ds = ray.data.from_pandas(data)
print("Ray Dataset:")
print(ray_ds)
print(f"\nSample:")
ray_ds.take(3)

### Transformations

In [ ]:
# SPARK: Transformations
spark_result = spark_df \
    .filter(F.col("value") > 0) \
    .withColumn("value_squared", F.col("value") ** 2) \
    .groupBy("category") \
    .agg(
        F.count("*").alias("count"),
        F.avg("value").alias("avg_value"),
        F.avg("value_squared").alias("avg_squared")
    )

print("Spark aggregation:")
spark_result.show()

In [ ]:
# RAY: Transformations
def transform_batch(batch: pd.DataFrame) -> pd.DataFrame:
    filtered = batch[batch["value"] > 0].copy()
    filtered["value_squared"] = filtered["value"] ** 2
    return filtered

ray_transformed = ray_ds.map_batches(transform_batch, batch_format="pandas")

# Aggregation
ray_agg = ray_transformed.groupby("category").mean(["value", "value_squared"])

print("Ray aggregation:")
ray_agg.take_all()

---

## 2. Performance Comparison

In [ ]:
# Larger dataset for meaningful comparison
n_large = 1_000_000
large_data = pd.DataFrame({
    "x": np.random.randn(n_large),
    "y": np.random.randn(n_large),
    "z": np.random.randn(n_large),
})

print(f"Large dataset: {len(large_data):,} rows, {large_data.memory_usage().sum() / 1e6:.1f} MB")

In [ ]:
# Test 1: Simple map operation
def benchmark_map():
    results = {}
    
    # Spark
    spark_large = spark.createDataFrame(large_data)
    start = time.time()
    spark_mapped = spark_large.withColumn("result", F.col("x") * 2 + F.col("y"))
    _ = spark_mapped.count()  # Force evaluation
    results["Spark"] = time.time() - start
    
    # Ray
    ray_large = ray.data.from_pandas(large_data)
    start = time.time()
    def map_func(batch):
        batch["result"] = batch["x"] * 2 + batch["y"]
        return batch
    ray_mapped = ray_large.map_batches(map_func, batch_format="pandas")
    _ = ray_mapped.count()  # Force evaluation
    results["Ray"] = time.time() - start
    
    return results

print("Benchmark: Map Operation")
print("="*40)
map_results = benchmark_map()
for framework, t in map_results.items():
    print(f"{framework}: {t:.3f}s")

In [ ]:
# Test 2: Aggregation
def benchmark_aggregation():
    results = {}
    
    # Create categorical data for groupby
    agg_data = large_data.copy()
    agg_data["group"] = np.random.choice(["A", "B", "C", "D", "E"], len(agg_data))
    
    # Spark
    spark_agg = spark.createDataFrame(agg_data)
    start = time.time()
    result = spark_agg.groupBy("group").agg(
        F.avg("x").alias("avg_x"),
        F.sum("y").alias("sum_y"),
        F.count("*").alias("count")
    )
    _ = result.collect()
    results["Spark"] = time.time() - start
    
    # Ray
    ray_agg = ray.data.from_pandas(agg_data)
    start = time.time()
    result = ray_agg.groupby("group").mean(["x", "y"])
    _ = result.take_all()
    results["Ray"] = time.time() - start
    
    return results

print("\nBenchmark: Aggregation")
print("="*40)
agg_results = benchmark_aggregation()
for framework, t in agg_results.items():
    print(f"{framework}: {t:.3f}s")

In [ ]:
# Test 3: Custom Python function (Ray's strength)
def benchmark_custom_udf():
    results = {}
    
    # Spark with UDF (slow due to serialization)
    from pyspark.sql.functions import udf
    from pyspark.sql.types import DoubleType
    
    @udf(DoubleType())
    def custom_func_spark(x, y, z):
        return float(np.sin(x) + np.cos(y) + np.sqrt(abs(z)))
    
    spark_udf = spark.createDataFrame(large_data)
    start = time.time()
    result = spark_udf.withColumn("custom", custom_func_spark("x", "y", "z"))
    _ = result.count()
    results["Spark UDF"] = time.time() - start
    
    # Ray with vectorized function (fast)
    def custom_func_ray(batch: pd.DataFrame) -> pd.DataFrame:
        batch["custom"] = np.sin(batch["x"]) + np.cos(batch["y"]) + np.sqrt(np.abs(batch["z"]))
        return batch
    
    ray_udf = ray.data.from_pandas(large_data)
    start = time.time()
    result = ray_udf.map_batches(custom_func_ray, batch_format="pandas")
    _ = result.count()
    results["Ray vectorized"] = time.time() - start
    
    return results

print("\nBenchmark: Custom Python Function")
print("="*40)
udf_results = benchmark_custom_udf()
for framework, t in udf_results.items():
    print(f"{framework}: {t:.3f}s")
print(f"\nSpeedup: {udf_results['Spark UDF']/udf_results['Ray vectorized']:.1f}x")

---

## 3. Feature Comparison

In [ ]:
comparison_table = pd.DataFrame({
    "Feature": [
        "Primary Use Case",
        "Programming Model",
        "SQL Support",
        "UDF Performance",
        "Stateful Computation",
        "ML Training",
        "Streaming",
        "GPU Support",
        "Fault Tolerance",
        "Language Support"
    ],
    "Spark": [
        "ETL, Data Processing",
        "DataFrame/RDD",
        "Excellent (Spark SQL)",
        "Slow (serialization)",
        "Limited",
        "MLlib (basic)",
        "Structured Streaming",
        "Rapids (limited)",
        "RDD lineage",
        "Scala, Python, Java, R"
    ],
    "Ray": [
        "ML, Custom Tasks",
        "Tasks/Actors",
        "Basic",
        "Fast (native Python)",
        "Actors",
        "Ray Train (excellent)",
        "Basic",
        "Native",
        "Object reconstruction",
        "Python (primary)"
    ]
})

print("Feature Comparison: Spark vs Ray")
print("="*80)
print(comparison_table.to_string(index=False))

---

## 4. When to Use Each Framework

In [ ]:
use_cases = {
    "USE SPARK WHEN": [
        "Complex SQL queries and joins",
        "ETL pipelines with structured data",
        "Data warehousing operations",
        "Integrating with Hive/HDFS ecosystem",
        "Need for data lineage and governance",
        "Heavy shuffle operations",
        "Existing Spark infrastructure"
    ],
    "USE RAY WHEN": [
        "ML model training (XGBoost, PyTorch, TF)",
        "Custom Python algorithms",
        "Stateful computation (actors)",
        "Real-time inference",
        "Reinforcement learning",
        "Hyperparameter tuning",
        "Need native Python performance"
    ],
    "USE BOTH WHEN": [
        "ETL with Spark → ML with Ray",
        "Complex data prep + model training",
        "Data validation (Spark) + Feature eng (Ray)",
        "Need best of both worlds"
    ]
}

print("Framework Selection Guide")
print("="*60)
for category, items in use_cases.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  • {item}")

---

## 5. Practical Decision Flowchart

In [ ]:
decision_flowchart = """
FRAMEWORK DECISION FLOWCHART
============================

Start: What's your primary task?
           │
           ├── SQL queries / Complex joins?
           │         │
           │         └── YES ──> SPARK
           │
           ├── ML Model Training?
           │         │
           │         └── YES ──> RAY
           │
           ├── Custom Python algorithms?
           │         │
           │         └── YES ──> RAY
           │
           ├── Batch ETL?
           │         │
           │         └── YES ──> SPARK
           │
           ├── Need stateful computation?
           │         │
           │         └── YES ──> RAY (Actors)
           │
           └── Both ETL and ML?
                     │
                     └── YES ──> BOTH (Spark ETL → Ray ML)
"""
print(decision_flowchart)

---

## 6. Exercise: Choose the Right Framework

For each scenario, decide whether to use Spark, Ray, or both:

In [ ]:
scenarios = [
    {
        "scenario": "Process 500GB of web logs, aggregate by user, join with user profiles",
        "answer": "SPARK - Complex joins and aggregations on large structured data"
    },
    {
        "scenario": "Train an XGBoost model on 100GB feature dataset",
        "answer": "RAY - ML training with Ray Train XGBoostTrainer"
    },
    {
        "scenario": "Build a recommendation system that maintains user state",
        "answer": "RAY - Stateful computation with Actors"
    },
    {
        "scenario": "ETL pipeline: clean data, create features, train model, serve predictions",
        "answer": "BOTH - Spark for ETL, Ray for ML training and serving"
    },
    {
        "scenario": "Real-time fraud detection with model inference",
        "answer": "RAY - Low-latency inference with Ray Serve"
    },
]

print("Framework Selection Exercises")
print("="*70)
for i, item in enumerate(scenarios, 1):
    print(f"\n{i}. {item['scenario']}")
    print(f"   → {item['answer']}")

---

## Summary

### Key Differences

| Aspect | Spark | Ray |
|--------|-------|-----|
| Strength | SQL, ETL, Joins | ML, Custom Python |
| Data Model | DataFrame/RDD | Tasks/Actors |
| State | Stateless transforms | Stateful actors |
| Python UDFs | Slow (serialization) | Fast (native) |

### Best Practice

**Don't choose one - use both!**

```
Spark ETL → Parquet → Ray ML → Model
```

### Next Steps

See `04c_data_handoff_patterns.ipynb` for data transfer strategies.

In [ ]:
# Cleanup
spark.stop()
ray.shutdown()
print("Cleanup complete.")